In [1]:
from pathlib import Path
import sys

import torch

# We'll use the CLIP vision model from Hugging Face transformers
from transformers import CLIPVisionModel, CLIPImageProcessor

# --- Helper to find project root (same pattern as other notebooks) ---

def find_project_root(start: Path) -> Path:
    """
    Walk upwards from `start` until we find a directory that has 'src' and 'data'.
    """
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise RuntimeError(f"Could not find project root starting from {start}")

cwd = Path.cwd()
project_root = find_project_root(cwd)
print("CWD         :", cwd)
print("PROJECT_ROOT:", project_root)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
print("project_root in sys.path:", str(project_root) in sys.path)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


/home/woody/iwi5/iwi5384h/software/private/conda/envs/emuru/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD         : /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/notebooks
PROJECT_ROOT: /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project
project_root in sys.path: True
Using device: cpu


In [2]:
# Choose a CLIP variant – ViT-B/32 is a good starting point
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"Loading CLIP vision model: {CLIP_MODEL_NAME}")

# Image processor (handles resize / normalization, etc.)
clip_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_NAME)

# Vision backbone (no text part)
clip_vision = CLIPVisionModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_vision.eval()

# Freeze all CLIP vision parameters for now (we'll only train a classifier head)
for p in clip_vision.parameters():
    p.requires_grad = False

print("\nLoaded CLIP vision backbone.")
print("Hidden size (feature dim):", clip_vision.config.hidden_size)
print("Expected image size:", clip_processor.size)
print("Do_resize:", clip_processor.do_resize, "| do_center_crop:", clip_processor.do_center_crop)


Loading CLIP vision model: openai/clip-vit-base-patch32

Loaded CLIP vision backbone.
Hidden size (feature dim): 768
Expected image size: {'shortest_edge': 224}
Do_resize: True | do_center_crop: True


In [3]:
import torch
from PIL import Image
import numpy as np

# Fake grayscale line image (e.g. 128 x 2048) just to check shape handling
H, W = 128, 1024
fake_arr = (np.random.rand(H, W) * 255).astype("uint8")
fake_img = Image.fromarray(fake_arr, mode="L")

# Convert to RGB for CLIP by duplicating channels
fake_img_rgb = fake_img.convert("RGB")

# Use CLIP's image processor to create pixel_values
inputs = clip_processor(
    images=fake_img_rgb,
    return_tensors="pt",
)

pixel_values = inputs["pixel_values"].to(device)  # (1, 3, 224, 224) usually
print("pixel_values shape:", tuple(pixel_values.shape))

with torch.no_grad():
    out = clip_vision(pixel_values)
    # pooler_output is usually the global representation (batch, hidden_size)
    feats = out.pooler_output  # (1, hidden_size)
    print("CLIP features shape:", tuple(feats.shape))


pixel_values shape: (1, 3, 224, 224)
CLIP features shape: (1, 768)
